# Example Analysis of Aligned Tables

This notebook demonstrates how to use the aligned tables generated by the table_aligner.py script for analysis.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set up paths
output_dir = Path("output")
if not output_dir.exists():
    print("Output directory not found. Please run align_tables.py first.")

## Load Aligned Tables

First, let's load all the aligned tables into a dictionary.

In [ ]:
# Load all aligned tables
aligned_tables = {}
for file_path in output_dir.glob("aligned_nf_*.csv"):
    file_id = file_path.stem.replace("aligned_nf_", "")
    aligned_tables[file_id] = pd.read_csv(file_path)

print(f"Loaded {len(aligned_tables)} aligned tables.")

## Examine a Single Table

Let's examine the structure of one of the aligned tables.

In [ ]:
# Choose a table to examine (e.g., nf_1)
table_id = "1"
if table_id in aligned_tables:
    df = aligned_tables[table_id]
    print(f"Table {table_id} shape: {df.shape}")
    
    # Show the first few rows
    df.head()

## Analyze Column Presence

Let's analyze which columns are present in which tables.

In [ ]:
# Initialize counters
all_columns = set()
pred_columns = set()
gt_columns = set()
common_columns = set()

# Process each table
for table_id, df in aligned_tables.items():
    # Get column base names (without _pred or _gt suffix)
    file_pred_cols = {col[:-5] for col in df.columns if col.endswith("_pred")}
    file_gt_cols = {col[:-3] for col in df.columns if col.endswith("_gt")}
    
    # Update sets
    pred_columns.update(file_pred_cols)
    gt_columns.update(file_gt_cols)
    all_columns.update(file_pred_cols.union(file_gt_cols))
    common_columns.update(file_pred_cols.intersection(file_gt_cols))

# Create result DataFrame
col_stats = pd.DataFrame({
    "Column": list(all_columns),
    "In_Pred": [col in pred_columns for col in all_columns],
    "In_GT": [col in gt_columns for col in all_columns],
    "In_Both": [col in common_columns for col in all_columns]
})

# Sort by presence in both, then GT, then pred
col_stats["Sort_Order"] = (col_stats["In_Both"].astype(int) * 4 + 
                          col_stats["In_GT"].astype(int) * 2 + 
                          col_stats["In_Pred"].astype(int))
col_stats = col_stats.sort_values(["Sort_Order", "Column"], ascending=[False, True])
col_stats = col_stats.drop(columns=["Sort_Order"])

# Display summary
print(f"Total unique columns: {len(all_columns)}")
print(f"Columns in predictions only: {len(pred_columns - gt_columns)}")
print(f"Columns in ground truth only: {len(gt_columns - pred_columns)}")
print(f"Common columns: {len(common_columns)}")

# Show the first few rows of column statistics
col_stats.head(10)

## Calculate Match Rates

Let's calculate how often the predicted values match the ground truth values for each column.

In [ ]:
# Initialize results
results = []

# Process each table
for table_id, df in aligned_tables.items():
    # Find common columns (those that have both _pred and _gt versions)
    base_cols = set()
    for col in df.columns:
        if col.endswith("_pred"):
            base_col = col[:-5]
            if f"{base_col}_gt" in df.columns:
                base_cols.add(base_col)
    
    # Calculate match rates for each column
    for base_col in base_cols:
        pred_col = f"{base_col}_pred"
        gt_col = f"{base_col}_gt"
        
        # Count matches (ignoring NaN values)
        valid_mask = (~df[pred_col].isna()) & (~df[gt_col].isna())
        if valid_mask.sum() == 0:
            match_rate = np.nan
        else:
            # Case-insensitive comparison
            matches = (df[pred_col][valid_mask].astype(str).str.lower() == 
                      df[gt_col][valid_mask].astype(str).str.lower())
            match_rate = matches.mean() * 100
        
        results.append({
            "File": table_id,
            "Column": base_col,
            "Match_Rate": match_rate,
            "Valid_Comparisons": valid_mask.sum(),
            "Total_Rows": len(df)
        })

# Create result DataFrame
match_rates = pd.DataFrame(results)

# Calculate average match rate across files for each column
summary = match_rates.groupby("Column").agg({
    "Match_Rate": ["mean", "min", "max", "count"],
    "Valid_Comparisons": "sum",
    "Total_Rows": "sum"
})
summary.columns = ["Avg_Match_Rate", "Min_Match_Rate", "Max_Match_Rate", 
                  "File_Count", "Valid_Comparisons", "Total_Rows"]
summary = summary.reset_index()
summary["Coverage"] = (summary["Valid_Comparisons"] / summary["Total_Rows"] * 100)
summary = summary.sort_values("Avg_Match_Rate", ascending=False)

# Show the top 10 columns with highest match rates
summary.head(10)

## Visualize Match Rates

Let's visualize the match rates for the top columns.

In [ ]:
# Filter out columns with NaN match rates
filtered_summary = summary.dropna(subset=["Avg_Match_Rate"])

# Get top 20 columns by match rate
top_columns = filtered_summary.head(20)

# Create bar chart
plt.figure(figsize=(12, 8))
sns.barplot(x="Avg_Match_Rate", y="Column", data=top_columns)
plt.title("Top 20 Columns by Match Rate")
plt.xlabel("Average Match Rate (%)")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

## Examine Specific Column Values

Let's examine the values for a specific column across all tables.

In [ ]:
# Choose a column to examine
column_to_examine = "dataType"  # Replace with a column of interest

# Collect values from all tables
pred_values = []
gt_values = []

for table_id, df in aligned_tables.items():
    pred_col = f"{column_to_examine}_pred"
    gt_col = f"{column_to_examine}_gt"
    
    if pred_col in df.columns:
        pred_values.extend(df[pred_col].dropna().tolist())
    
    if gt_col in df.columns:
        gt_values.extend(df[gt_col].dropna().tolist())

# Count unique values
pred_counts = pd.Series(pred_values).value_counts()
gt_counts = pd.Series(gt_values).value_counts()

# Display results
print(f"Column: {column_to_examine}")
print("\nPredicted values:")
print(pred_counts)
print("\nGround truth values:")
print(gt_counts)

## Compare Values for a Specific File and Column

Let's compare the predicted and ground truth values for a specific column in a specific file.

In [ ]:
# Choose a file and column to examine
file_to_examine = "1"  # Replace with a file ID of interest
column_to_examine = "dataType"  # Replace with a column of interest

if file_to_examine in aligned_tables:
    df = aligned_tables[file_to_examine]
    pred_col = f"{column_to_examine}_pred"
    gt_col = f"{column_to_examine}_gt"
    
    if pred_col in df.columns and gt_col in df.columns:
        # Create a comparison DataFrame
        comparison = pd.DataFrame({
            "Predicted": df[pred_col],
            "Ground Truth": df[gt_col],
            "Match": df[pred_col].astype(str).str.lower() == df[gt_col].astype(str).str.lower()
        })
        
        # Display the comparison
        print(f"Comparison for file {file_to_examine}, column {column_to_examine}:")
        print(comparison)
        
        # Calculate match rate
        valid_mask = (~df[pred_col].isna()) & (~df[gt_col].isna())
        if valid_mask.sum() > 0:
            matches = (df[pred_col][valid_mask].astype(str).str.lower() == 
                      df[gt_col][valid_mask].astype(str).str.lower())
            match_rate = matches.mean() * 100
            print(f"\nMatch rate: {match_rate:.2f}%")
    else:
        print(f"Column {column_to_examine} not found in both predicted and ground truth for file {file_to_examine}.")
else:
    print(f"File {file_to_examine} not found.")

## Conclusion

This notebook demonstrates how to analyze the aligned tables generated by the table_aligner.py script. You can use these techniques to perform more detailed analysis of the data, such as:

- Identifying columns with high or low match rates
- Examining specific values that don't match
- Analyzing patterns in the data
- Generating reports on the quality of the predictions

For more advanced analysis, you can use the `analyze_aligned_tables.py` script, which provides additional functionality for analyzing the aligned tables.